In [1]:
import sqlite3
import json
import MyTools as MT
import pandas as pd
import Variable as V
from importlib import reload
from datetime import datetime
import numpy as np
import sql_code as SQL

In [2]:
def execute_sql_script_by_id(ids, nm):
    print(f"Выполняется код: {nm}")  # Выводим часть скрипта для наглядности
    MT.gp_execute(sql_script = ids)
    #print (ids)

In [3]:
def calculate (tb_from, tb_rule, tb_rezult):
    def is_number(value):
        try:
            float(value)
            return True
        except (ValueError, TypeError):
            return False
    
    # Get the column names of t_qimen and t_spr_rule
    qimen_info = MT.gp_execute(sql_script = f"PRAGMA table_info({tb_from});", rezim = 'fetchall')
    qimen_columns = [info[1] for info in qimen_info]
    
    spr_rule_info = MT.gp_execute(sql_script = f"PRAGMA table_info({tb_rule});", rezim = 'fetchall')
    spr_rule_columns = [info[1] for info in spr_rule_info]

    # Extra columns in t_spr_rule
    extra_columns = set(spr_rule_columns) - set(qimen_columns)

    spr_rule_rows = MT.gp_execute(sql_script = F"SELECT * FROM {tb_rule};", rezim = 'fetchall')
    spr_rule_column_indices = {col: idx for idx, col in enumerate(spr_rule_columns)}

    
    MT.gp_execute(sql_script=f'DELETE FROM {tb_rezult}')
    MT.gp_execute(sql_script=f'DELETE FROM rule_calculate_log')
    MT.gp_execute(sql_script='VACUUM')
    total_kol_rule = len(spr_rule_rows)
    print(f'Всего будет обработано {total_kol_rule:,} правил')
    for num, rule_row in enumerate(spr_rule_rows):
        
        # Build condition dictionary excluding extra columns and empty values
        condition = {}
        for col in qimen_columns:
            try:
                idx = spr_rule_column_indices[col]
            except:
                continue
            value = rule_row[idx]
            if value is not None and value != '':
                condition[col] = value
    
        if not condition:
            continue  # Skip if no conditions to apply
        # Build WHERE clause and parameters
        where_clauses = []
        params = []
        for col, value in condition.items():
            if is_number(value):
                where_clauses.append(f"{col} = {value}")
            else:
                # Экранируем одиночные кавычки в значении
                value_escaped = str(value).replace("'", "''")
                where_clauses.append(f"{col} = '{value_escaped}'")
                    
        where_clause = ' AND '.join(where_clauses)
    
        # Select matching rows from t_qimen
        query = f"SELECT Dt, type_rasklada, name_hour, palace  FROM {tb_from} WHERE {where_clause};"       
        matching_rows = MT.gp_execute (sql_script = query, rezim = 'fetchall')
    
        # Get extra columns values from rule_row
        gr = rule_row[spr_rule_column_indices['gr']]
        id_combination = rule_row[spr_rule_column_indices['id']]
        nm = rule_row[spr_rule_column_indices['nm']]
        describe = rule_row[spr_rule_column_indices['des']]
        coef = rule_row[spr_rule_column_indices['coef']]
        flag_stop = rule_row[spr_rule_column_indices['flag_stop']]

        #Записываем лог
        current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        MT.gp_execute(sql_script= f"""
        insert into rule_calculate_log
        values ('{current_time}', {id_combination}, "{query}")
        """)
        
        # Prepare condition JSON
        condition_json = json.dumps(condition, ensure_ascii=False)
    
        # Подготавливаем данные для вставки
        insert_data = [
            (Dt, type_rasklada, name_hour, palace, gr, id_combination, nm, describe, coef, flag_stop, condition_json)
            for Dt, type_rasklada, name_hour, palace in matching_rows
        ]
        
        insert_query = f""" 
        INSERT INTO {tb_rezult} (dt, type_rasklada, name_hour, palace, gr, id, nm, des, coef, flag_stop, condition)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
            """
        conn = sqlite3.connect('BD.db')  # Замените на путь к вашей базе данных
        cursor = conn.cursor()
        cursor.executemany(insert_query, insert_data)
    
        if num%200 == 0:
            now = datetime.now()
            current_datetime = now.strftime("%d.%m.%Y %H:%M")
            print  (f'{current_datetime} Обработано {num} правил или {round(num/total_kol_rule*100,2)}% от всего количества')
      # Фиксируем транзакцию
        conn.commit()
        conn.close()

In [4]:
MT.gp_execute(sql_script="""update t_spr_rule set activation_stop_factor = null where activation_stop_factor ="" """)
MT.gp_execute(sql_script="""update t_spr_rule set walk_stop_factor = null where walk_stop_factor ="" """)

In [5]:
calculate (tb_from = 'v_qimen', tb_rule = 'v_spr_rule_walk', tb_rezult = 't_rezult_analyza')

Всего будет обработано 2,828 правил
09.11.2025 13:44 Обработано 0 правил или 0.0% от всего количества
09.11.2025 13:45 Обработано 200 правил или 7.07% от всего количества
09.11.2025 13:46 Обработано 400 правил или 14.14% от всего количества
09.11.2025 13:47 Обработано 600 правил или 21.22% от всего количества
09.11.2025 13:48 Обработано 800 правил или 28.29% от всего количества
09.11.2025 13:50 Обработано 1000 правил или 35.36% от всего количества
09.11.2025 13:51 Обработано 1200 правил или 42.43% от всего количества
09.11.2025 13:52 Обработано 1400 правил или 49.5% от всего количества
09.11.2025 13:54 Обработано 1600 правил или 56.58% от всего количества
09.11.2025 13:56 Обработано 1800 правил или 63.65% от всего количества
09.11.2025 13:58 Обработано 2000 правил или 70.72% от всего количества
09.11.2025 14:00 Обработано 2200 правил или 77.79% от всего количества
09.11.2025 14:02 Обработано 2400 правил или 84.87% от всего количества
09.11.2025 14:03 Обработано 2600 правил или 91.94% о

# Делаем расчеты

In [6]:
def сalculating_impact ():
    def chek_null (column, id):
        try:
            pivot_df[column] = pivot_df[id]
        except:
            print(f'Для колонки {column} нет показателя {id}')
            pivot_df[column] = 0

    sql = """select 
        dt
    , type_rasklada
    , name_hour
    , palace
    , gr
    , id
    , coef
    , coalesce(flag_stop,0) as flag_stop
    from t_rezult_analyza
    where 1 = 1 
    and dt>= date('now')
    """
    
    col = [
        'dt'
    , 'type_rasklada'
    , 'name_hour'
    , 'palace'
    , 'gr'
    , 'id'
    , 'coef'
    , 'flag_stop'
    ]
    
    col_dtype = {
        'type_rasklada':'int64'
        , 'name_hour':'str'
    , 'palace':'int64'
    , 'gr':'int64'
    , 'id':'int64'
    , 'coef':'float64'
    , 'flag_stop':'int64'
    }
    
    conn = sqlite3.connect('BD.db') 
    cursor = conn.cursor()
    
    df = pd.read_sql_query(sql, conn, parse_dates=['dt'], dtype=col_dtype)
    conn.close()
    
    # Оставляем события после текущей даты расчета
    df['dt'] = df['dt'].dt.normalize()  # нормализуем до даты без времени
    # Получаем текущую дату
    current_date = pd.Timestamp('today').normalize()
    # Фильтрация DataFrame
    filtered_df = df[df['dt'] >= current_date]
    filtered_df['day_of_week'] = filtered_df['dt'].dt.strftime('%a')
    
    # Добавляем дни недели к датам
    day_translation = {
        'Mon': 'Пн.',
        'Tue': 'Вт.',
        'Wed': 'Ср.',
        'Thu': 'Чт.',
        'Fri': 'Пт.',
        'Sat': 'Сб.',
        'Sun': 'Вс.'
    }
    filtered_df['day_of_week'] = filtered_df['day_of_week'].map(day_translation)
    
    # Переводим в перекрестный вид
    pivot_table = filtered_df.pivot_table( values = 'coef'
                                    , index = ['dt', 'day_of_week', 'type_rasklada', 'name_hour', 'palace']
                                    , columns = ['gr']
                                    , aggfunc='sum'
                                    , fill_value=0)
    pivot_df = pivot_table.reset_index()
    
    # Считаем отдельно колонку со стопами и добавляем к перекрестной таблице
    flag_stop = filtered_df.groupby(['dt', 'day_of_week', 'type_rasklada', 'name_hour', 'palace'])['flag_stop'].sum()
    flag_stop= flag_stop.reset_index()
    pivot_df = pivot_df.merge(flag_stop, on=['dt', 'day_of_week', 'type_rasklada', 'name_hour', 'palace'], validate = 'one_to_one').reset_index(drop=True)
    
    # Рассчитываем компоненты формулы
    pivot_df['sector'] = pivot_df['palace']
    pivot_df['palace'] = pivot_df[122030203]
    pivot_df['gate'] = (pivot_df[122030101] * pivot_df[122030102] * pivot_df[122030103] * np.where(pivot_df[122030104] == 0, 1, pivot_df[122030104]))
    
    pivot_df['stryctyra'] = (
        (pivot_df[122010201] * pivot_df[122010202] * pivot_df[122010203] 
        +
        pivot_df[122010101] * pivot_df[122010102] * pivot_df[122010203]
        +
        pivot_df[222000101] 
        )
        * pivot_df[122010000]
    )
    
    pivot_df['spirit'] = (pivot_df[122030301] * pivot_df[122030302] * pivot_df[122030303])
    pivot_df['star'] = (pivot_df[122030401] * pivot_df[122030402] * pivot_df[122030403])
    pivot_df['combination'] = pivot_df[122020000]
    
    chek_null(column = 'napravl_chas_den', id = 122060100)
    chek_null(column = 'napravl_chas_den_mes', id = 122060200)
    chek_null(column = 'napravl_chas_den_mes_god', id = 122060300)
    
    pivot_df['five_god'] = pivot_df[122090100]
    pivot_df['five_mes'] = pivot_df[122090200]
    pivot_df['pusto_den'] = pivot_df[122080300]
    
    chek_null(column = 'pusto_hour', id = 122080400)
    chek_null(column = 'palace_with_my_palace', id = 222050000)
    chek_null(column = 'scdg_good_time', id = 211040400)
    
    # Расчет благоприятности дворца / направления
    pivot_df['coef'] = (
        np.where (pivot_df['flag_stop'] == 0,
            pivot_df['gate'] 
            * 
            np.where((pivot_df['stryctyra'] + pivot_df['star'] + pivot_df['combination']+ pivot_df['spirit']) < 0, 
                     0, 
                     (pivot_df['stryctyra'] + pivot_df['star'] + pivot_df['combination']+ pivot_df['spirit']))
            
            * np.where(pivot_df['pusto_hour'] > 0, pivot_df['pusto_hour'], 1)
            * np.where(pivot_df['pusto_den'] > 0, pivot_df['pusto_den'], 1)
        
            + pivot_df['five_god']
            + pivot_df['five_mes']
            + pivot_df['scdg_good_time']
            + pivot_df['napravl_chas_den']
            + pivot_df['napravl_chas_den_mes']
            + pivot_df['napravl_chas_den_mes_god']
        ,0)
    )

    # Извлекаем год из даты
    pivot_df['year'] = pivot_df['dt'].dt.year
    
    # Находим максимальный coef для каждого года
    max_year_coef = pivot_df.groupby('year')['coef'].transform(lambda x: x[x > 0].quantile(0.95) if x[x > 0].size > 0 else None)

    # Добавляем max_coef в pivot_df
    pivot_df['max_year_coef'] = max_year_coef
    
    # Рассчитываем долю coef от максимального coef в рамках каждого года
    pivot_df['coef_year_share'] = round(pivot_df['coef'] / max_year_coef,4)*100

    # Удаляем временный столбец year, если он не нужен
    #pivot_df = pivot_df.drop(columns=['year'])
    
    # Считаем силу прогулки в рамках месяца
    # Добавляем колонку для месяца и года
    pivot_df['month'] = pivot_df['dt'].dt.to_period('M')
    
    # Находим 95-й персентиль для каждого месяца
    percentile_95 = pivot_df[pivot_df['coef'] > 0].groupby('month')['coef'].quantile(0.95).reset_index()
    percentile_95.columns = ['month', 'max_month_coef']
    # Объединяем DataFrame с 95-ым персентилем
    pivot_df = pivot_df.merge(percentile_95, on='month')
    # Рассчитываем процентное значение от 95-го персентиля
    pivot_df['coef_month_share'] = (pivot_df['coef'] / pivot_df['max_month_coef']) * 100
    
    # Расчет благоприятности для меня
    pivot_df['total_coef'] = pivot_df['coef'] * pivot_df['palace_with_my_palace']
    
    #Записываем в базу
    conn = sqlite3.connect('BD.db')  # Замените на путь к вашей базе данных
    cursor = conn.cursor()
    # Преобразование колонки dt в формат даты
    pivot_df['dt'] = pivot_df['dt'].dt.date
    pivot_df['month'] = pivot_df['month'].astype(str)
    pivot_df.to_sql('t_walk_analyze', conn, if_exists='replace', index=False)
    conn.close()

In [7]:
execute_sql_script_by_id(ids =  SQL.sql_2220001010001, nm = '2220001010001 Оценка редкости структур') # Оценка редкости структур
execute_sql_script_by_id(ids =  SQL.sql_2110404000001, nm = '2110404000001 СКДГ. Хорошее время') # записали приличное время по СКДГ
execute_sql_script_by_id(ids =  SQL.sql_1220200000793, nm = '1220200000793 4 Тайных земных дверей') # 4 Тайных земных дверей

Выполняется код: 2220001010001 Оценка редкости структур
Выполняется код: 2110404000001 СКДГ. Хорошее время
Выполняется код: 1220200000793 4 Тайных земных дверей


In [8]:
сalculating_impact()

Для колонки napravl_chas_den нет показателя 122060100
Для колонки napravl_chas_den_mes нет показателя 122060200
Для колонки napravl_chas_den_mes_god нет показателя 122060300


KeyError: 122090100

# Записываем совпадение направления день - месяц - год

In [9]:
reload(SQL)
execute_sql_script_by_id(ids =  SQL.sql_1220601000001, nm = '1220601000001 Совпадение направлений. Час = День') # записали совпадение направления часа и дня
execute_sql_script_by_id(ids =  SQL.sql_1220602000001, nm = '1220602000001 Совпадение направлений. Час = День = Месяц') # записали совпадение направления часа и дня и месяца
execute_sql_script_by_id(ids =  SQL.sql_1220603000001, nm = '1220603000001 Совпадение направлений. Совпадение направлений. Час = День = Месяц = Год') # записали совпадение направления часа и дня и месяца и года
execute_sql_script_by_id(ids = SQL.sql_2220500000001, nm = '2220500000001 Цимень. Взаимодейтсвие дворцов с дворцом, где мой элемент') # записали связь с дворцом моего элемента

Выполняется код: 1220601000001 Совпадение направлений. Час = День
Выполняется код: 1220602000001 Совпадение направлений. Час = День = Месяц
Выполняется код: 1220603000001 Совпадение направлений. Совпадение направлений. Час = День = Месяц = Год
Выполняется код: 2220500000001 Цимень. Взаимодейтсвие дворцов с дворцом, где мой элемент


Руками пробежать блок расчетов пока

In [10]:
сalculating_impact ()

Для колонки napravl_chas_den_mes нет показателя 122060200
Для колонки napravl_chas_den_mes_god нет показателя 122060300


KeyError: 122090100

# Агрегируем описание прогулок

In [ ]:
import sqlite3
def accumulate_and_save_descriptions(db_path):
    # Подключение к базе данных SQLite
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Создание таблицы t_walk_description, если она не существует
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS t_walk_description (
        Dt TEXT,
        name_hour TEXT,
        palace INTEGER,
        description TEXT
    )
    """)
    
    # Очистка таблицы перед новой записью
    cursor.execute("DELETE FROM t_walk_description")

    # Запрос для извлечения данных из таблицы t_rezult_analyza
    query = """
    SELECT Dt, name_hour, palace, condition, nm, des, coef, flag_stop
    FROM t_rezult_analyza
    where 1 = 1
    and gr in (122010000, 122020000, 122030101, 122030104, 122030301, 122030401, 122060100, 122060200, 122060300, 122080300, 122080400,
    122090100, 122090200, 211040400, 222000101, 222050000, 232000000)
    """
    
    cursor.execute(query)
    
    # Создаем словарь для аккумулирования описаний
    result = {}
    
    # Проходимся по всем полученным строкам
    for row in cursor.fetchall():
        dt, name_hour, palace, condition, nm, des, coef, flag_stop = row
        
        # Форматируем описание для текущего показателя согласно маске
        description = f"{condition} {nm}. {des}. Влияние = {coef}. Стоп-фактор = {flag_stop}"
        
        # Создаем ключ для группировки
        key = (dt, name_hour, palace)
        
        # Если ключ еще не существует в результате, создаем новый список
        if key not in result:
            result[key] = []
        
        # Добавляем описание к списку
        result[key].append(description)
    
    # Заполняем таблицу t_walk_description
    for key, descriptions in result.items():
        aggregated_description = "\n".join(descriptions)
        dt, name_hour, palace = key
        
        # Вставляем данные в таблицу
        cursor.execute("""
        INSERT INTO t_walk_description (Dt, name_hour, palace, description)
        VALUES (?, ?, ?, ?)
        """, (dt, name_hour, palace, aggregated_description))

    # Сохраняем изменения и закрываем соединение с базой данных
    conn.commit()
    conn.close()

# Использование функции
db_path = 'BD.db'  # Замените на путь к вашей базе данных
accumulate_and_save_descriptions(db_path)

# Записываем в календарь

## Прогулки индивидуальные

In [2]:
sql = """select dt, name_hour, moscow_hour_recomend, moscow_hour_st, moscow_hour_en, sector, direction, total_coef, coef_month_share , coef_year_share, description   from v_ideal_walk_hour
where coef_month_share >= 15"""
query = MT.gp_execute(sql_script = sql, rezim = 'list')
df_walk = pd.DataFrame(query, columns=['dt',
                                       'name_hour'
                                       , 'moscow_hour_recomend'
                                       , 'moscow_hour_st'
                                       , 'moscow_hour_en'
                                       , 'sector'
                                       , 'direction'
                                       , 'total_coef'
                                       , 'coef_month_share'
                                       , 'coef_year_share'
                                       , 'description'
                                      ])

In [3]:
df_walk['id'] = [hash( 
                            str(row['dt']) 
                            + str(row['name_hour'])
                            + str(row['sector'])
                           ) for index, row in df_walk.iterrows()]
df_walk['id'] = df_walk['id'].astype(str)

In [4]:
df_walk['dt_start'] = [f"{row['dt']} {row['moscow_hour_st']}" for index, row in df_walk.iterrows()]
df_walk['dt_end'] = [f"{row['dt']} {row['moscow_hour_en']}" for index, row in df_walk.iterrows()]
df_walk['calendar'] = 'Прогулки_индивидуальные'
df_walk['title'] = [f"Прогулка на {row['direction']} - {round(row['coef_month_share'],2)}% / {round(row['coef_year_share'],2)}%" for index, row in df_walk.iterrows()]
df_walk['comment'] = df_walk['description']

In [5]:
# Записываем прогулки в таблицу Event
#MT.gp_execute(sql_script="delete from event  where 1 = 1 and dt_end < date('now')")# Удаляем старые / прошедшие записи
MT.gp_execute(sql_script="delete from event where calendar = 'Прогулки_индивидуальные'")# Удаляем старые прогулки

conn = sqlite3.connect(V.bd_name)
# Указание колонок, которые нужно записать в базу данных
columns_to_write = ['id', 'dt_start', 'dt_end', 'calendar', 'title', 'comment']
# Запись только указанных колонок в таблицу "my_table"
df_walk[columns_to_write].to_sql('event', conn, index=False, if_exists='append')
# Закрытие соединения
conn.close()

## Прогулки общие

In [6]:
sql = """select dt, name_hour, moscow_hour_recomend, moscow_hour_st, moscow_hour_en, sector, direction, total_coef, coef, coef_month_share , coef_year_share,description
from v_good_walk_hour
where 1 = 1
    and total_coef = 0
    and coef_month_share >=25"""
query = MT.gp_execute(sql_script = sql, rezim = 'list')
df_walk = pd.DataFrame(query, columns=['dt',
                                       'name_hour'
                                       , 'moscow_hour_recomend'
                                       , 'moscow_hour_st'
                                       , 'moscow_hour_en'
                                       , 'sector'
                                       , 'direction'
                                       , 'total_coef'
                                       , 'coef'
                                       , 'coef_month_share'
                                       , 'coef_year_share'
                                       , 'description'
                                      ])

In [7]:
df_walk['id'] = [hash( 
                            str(row['dt']) 
                            + str(row['name_hour'])
                            + str(row['sector'])
                           ) for index, row in df_walk.iterrows()]
df_walk['id'] = df_walk['id'].astype(str)

In [8]:
df_walk['dt_start'] = [f"{row['dt']} {row['moscow_hour_st']}" for index, row in df_walk.iterrows()]
df_walk['dt_end'] = [f"{row['dt']} {row['moscow_hour_en']}" for index, row in df_walk.iterrows()]
df_walk['calendar'] = 'Прогулки_общие'
df_walk['title'] = [f"Прогулка на {row['direction']} - {round(row['coef_month_share'],2)}% / {round(row['coef_year_share'],2)}%" for index, row in df_walk.iterrows()]
df_walk['comment'] = df_walk['description']

In [9]:
# Записываем прогулки в таблицу Event
#MT.gp_execute(sql_script="delete from event  where 1 = 1 and dt_end < date('now')")# Удаляем старые / прошедшие записи
MT.gp_execute(sql_script="delete from event where calendar = 'Прогулки_общие'")# Удаляем старые прогулки

conn = sqlite3.connect(V.bd_name)
# Указание колонок, которые нужно записать в базу данных
columns_to_write = ['id', 'dt_start', 'dt_end', 'calendar', 'title', 'comment']
# Запись только указанных колонок в таблицу "my_table"
df_walk[columns_to_write].to_sql('event', conn, index=False, if_exists='append')
# Закрытие соединения
conn.close()

# Обновляем календарь Mail

In [2]:
reload(MT)
MT.write_calendar()

['Основной', 'Активации', 'Ведические события', 'Дни рождений', 'Духи', 'Ключевые события', 'Манифестации', 'Прогулки_индивидуальные', 'Прогулки_общие', 'Удачное_время', 'Телепрограмма', 'Работа', 'Личное']
[ERROR] Не удалось создать UID=-4540204066388868012 в Активации
[ERROR] Не удалось создать UID=1249649495115445918 в Активации
[ERROR] Не удалось создать UID=9099000515192692517 в Удачное_время
[ERROR] Не удалось создать UID=460965591569880487 в Удачное_время
[ERROR] Не удалось создать UID=-4704178584037905405 в Удачное_время
{'Активации': {'created': 30, 'updated': 0, 'deleted': 0, 'unchanged': 0}, 'Манифестации': {'created': 15, 'updated': 0, 'deleted': 0, 'unchanged': 0}, 'Удачное_время': {'created': 42, 'updated': 0, 'deleted': 0, 'unchanged': 0}}
Скрипт write_calendar выполнился за 0 дней 0 часов 1 минут 12 секунд (с 2025-12-24 13:09:34 по 2025-12-24 13:10:46)
